# 14: Brapi (cotação, perfil e conferência do ticker)

Até aqui, todo dado de mercado vinha do **yfinance**, uma biblioteca que lê o site do Yahoo Finance. Ela funciona
bem, mas tem três problemas para este projeto:

- **Não é uma API oficial.** O Yahoo não promete nada: um campo pode mudar de nome ou sumir sem aviso.
- **O setor vem em inglês e na classificação do Yahoo** ("Specialty Industrial Machinery"), não na da B3, que é a
  que o investidor brasileiro conhece ("Bens Industriais").
- **Ticker que não existe só aparece no meio da análise.** Quem digita `XXXX3` passa pela validação do formato
  (4 letras e 1 número) e o grafo começa a rodar: o Supervisor chama o LLM, o YFinance não acha nada, e créditos
  foram gastos à toa.

A **Brapi** (brapi.dev) é uma API brasileira de dados da bolsa. No plano gratuito, ela entra no projeto em três
pontos: confere se o ticker existe **antes** do grafo rodar, traz a cotação e o perfil da empresa em português
e mostra os maiores do setor. O yfinance continua como reserva e como fonte dos demonstrativos.

Este notebook não chama nenhum LLM: as chamadas à Brapi e ao yfinance são gratuitas.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**Brapi:** API paga por assinatura, com um plano gratuito, que reúne cotações da B3 e dados das empresas abertas.
Os demonstrativos das empresas brasileiras (balanço, resultado, fluxo de caixa) são entregues por obrigação à
**CVM** (Comissão de Valores Mobiliários, o órgão que fiscaliza o mercado de capitais): o formulário trimestral se
chama ITR e o anual, DFP. São públicos, e APIs como a Brapi os organizam num formato fácil de ler.

**Plano gratuito x pago:** o gratuito libera a cotação, o perfil da empresa (setor, subsetor e um resumo do negócio)
e a lista de papéis com busca e filtro por setor. Demonstrativos, dividendos e indicadores como P/L e ROE ficam nos
planos pagos e voltam com erro **403** (acesso negado). Por isso Piotroski, trimestrais, anomalias e métricas de
risco continuam no yfinance.

**Ticker de sandbox:** alguns tickers (PETR4, VALE3, MGLU3, ITUB4) respondem com **todos** os módulos, mesmo sem
token. Servem para quem está testando a API. Cuidado: testar o plano com a VALE3 dá a impressão de que tudo está
liberado. Com a WEGE3, um ticker comum, aparece o limite real.

**Cotação x demonstrativo:** a cotação é o preço do papel, que muda a cada negócio na bolsa. O demonstrativo é o
retrato contábil da empresa num período (trimestre ou ano) e só muda quando ela publica um novo.

**Faixa de 52 semanas:** o menor e o maior preço do último ano. Mostra onde o preço de hoje está em relação ao ano:
perto da mínima, perto da máxima ou no meio.

**Valor de mercado:** preço da ação vezes o número de ações. É quanto a bolsa diz que a empresa inteira vale hoje.

**LPA (lucro por ação):** o lucro dos últimos 12 meses dividido pelo número de ações. Daí sai o P/L:
**P/L = preço / LPA**, quantos anos daquele lucro "pagariam" o preço de uma ação.

**Setor e subsetor da B3:** a classificação que a própria bolsa usa (ex: setor "Bens Industriais", subsetor
"Máquinas e Equipamentos"). Agrupa empresas que vivem dos mesmos clientes e dos mesmos ciclos.

**Por que comparar com os pares do setor:** um número sozinho não diz muito. Um P/L de 30 pode ser caro para um
banco e normal para uma empresa de tecnologia industrial. Ver os maiores do setor, e a posição da empresa entre
eles, dá a régua para o Analyzer.

## 2. O ticker existe?

`ticker_existe` devolve três respostas, não duas:

- `True`: a Brapi devolveu a cotação.
- `False`: a Brapi respondeu **404** (não encontrado). Só este caso faz a entrada recusar o ticker.
- `None`: não deu para conferir (sem token, cota estourada, timeout, Brapi fora do ar). A análise segue, e o
  yfinance decide como antes.

As entradas (CLI, dashboard, e2e) chamam isso depois de validar o formato e antes de rodar o grafo.

In [2]:
from fluxo_financeiro.tools.brapi import mensagem_ticker_inexistente, ticker_existe

for ticker in ["WEGE3", "XXXX3"]:
    print(f"{ticker}: ticker_existe -> {ticker_existe(ticker)}")

WEGE3: ticker_existe -> True
XXXX3: ticker_existe -> False


Quando o ticker não existe, a mensagem tenta sugerir um parecido: busca as mesmas 4 letras na lista da Brapi e
pega o primeiro papel inteiro diferente do digitado. A sugestão não é aceita sozinha: a pessoa digita de novo.

In [3]:
print(mensagem_ticker_inexistente("WEGE4"))
print(mensagem_ticker_inexistente("XXXX3"))

Ticker não encontrado na B3: WEGE4. Você quis dizer WEGE3?
Ticker não encontrado na B3: XXXX3.


## 3. O perfil da empresa

`buscar_perfil` faz uma chamada só: a cotação com o módulo `summaryProfile` junto. Devolve um `PerfilBrapi`, ou
`None` se a Brapi falhar (nunca levanta exceção).

In [4]:
from fluxo_financeiro.tools.brapi import buscar_perfil

perfil = buscar_perfil("WEGE3")
for campo in ("nome", "preco", "variacao_pct", "minima_52s", "maxima_52s", "valor_mercado", "lpa", "setor", "subsetor"):
    print(f"{campo:14} {getattr(perfil, campo)!r}")
print(f"{'sumario':14} {perfil.sumario[:300]!r}...")

nome           'WEG SA'
preco          48.19
variacao_pct   -0.23
minima_52s     41.15
maxima_52s     54.53
valor_mercado  210010383322.0
lpa            1.4940624
setor          'Bens Industriais'
subsetor       'Máquinas e Equipamentos'
sumario        'A WEG S.A. é uma companhia brasileira de equipamentos eletroeletrônicos fundada em 1961, com sede em Jaraguá do Sul (SC), e listada na B3 sob o ticker WEGE3. A empresa atua no segmento de bens de capital industriais e desenvolve, fabrica e comercializa motores elétricos, geradores, transformadores, '...


**P/L = preço / LPA, na prática.** A Brapi do plano gratuito não manda o P/L (o campo vem vazio na WEGE3), mas
manda o preço e o LPA. A conta abaixo compara com o P/L que o yfinance informa.

In [5]:
import yfinance as yf

pl_conta = perfil.preco / perfil.lpa
pl_yahoo = yf.Ticker("WEGE3.SA").info.get("trailingPE")
print(f"Preço R$ {perfil.preco:.2f} / LPA R$ {perfil.lpa:.2f} = P/L {pl_conta:.1f}")
print(f"P/L do yfinance: {pl_yahoo:.1f}" if pl_yahoo else "P/L do yfinance: indisponível")

Preço R$ 48.19 / LPA R$ 1.49 = P/L 32.3
P/L do yfinance: 32.3


## 4. Os maiores do setor

A lista de papéis da Brapi aceita um filtro por setor e ordena por valor de mercado. Repare que ela usa outra
classificação, em inglês ("Producer Manufacturing"), diferente do setor em português do perfil. A lista também traz
o papel fracionário (`WEGE3F`, usado para comprar menos de 100 ações) e mais de uma classe da mesma empresa
(ações ON e PN, como SHUL3 e SHUL4). `buscar_pares_do_setor` fica com um papel inteiro por empresa.

In [6]:
from fluxo_financeiro.tools.brapi import buscar_pares_do_setor

pares = buscar_pares_do_setor("WEGE3")
print(f"Setor na lista: {pares.setor} | WEGE3 em {pares.posicao}º de {pares.total}")
for i, (codigo, nome, valor) in enumerate(pares.pares, start=1):
    print(f"  {i}. {codigo:7} {nome:35} R$ {valor / 1e9:.1f} bi")

Setor na lista: Producer Manufacturing | WEGE3 em 1º de 27
  1. WEGE3   WEG S.A.                            R$ 210.0 bi
  2. FRAS3   FRASLE MOBILITY S.A.                R$ 5.9 bi
  3. SHUL4   SCHULZ S.A.                         R$ 5.4 bi
  4. DXCO3   DEXCO S.A.                          R$ 4.3 bi
  5. POMO4   MARCOPOLO S.A.                      R$ 4.3 bi


## 5. O limite do plano gratuito: demonstrativos dão 403

Pedindo os resultados trimestrais da WEGE3, a Brapi recusa. A célula imprime só o tipo e o código do erro.

In [7]:
from fluxo_financeiro.tools.brapi import criar_cliente

try:
    criar_cliente().quote.retrieve(tickers="WEGE3", modules="incomeStatementHistoryQuarterly")
    print("Os demonstrativos vieram: o token tem acesso a esse módulo.")
except Exception as e:
    codigo = e.body.get("code") if isinstance(getattr(e, "body", None), dict) else None
    print(f"{type(e).__name__} | HTTP {getattr(e, 'status_code', None)} | código: {codigo}")

PermissionDeniedError | HTTP 403 | código: MODULES_NOT_AVAILABLE


## 6. A tool do YFinance, com as duas fontes

`buscar_dados_acao_b3` continua com o mesmo nome (o agente a chama por ele), mas agora junta as duas fontes:

| Linha da saída | Fonte |
|---|---|
| Empresa, Preço Atual (com a variação no dia), Faixa de 52 semanas, Valor de mercado, LPA | Brapi |
| P/L, ROE, DY | yfinance |
| Setor e Subsetor | Brapi (o yfinance dá a "Industria" em inglês, de reserva) |
| Maiores do setor | Brapi (lista por setor) |
| Sumário de Negócio | Brapi, em português (o yfinance, em inglês, de reserva) |
| Histórico dos 5 últimos fechamentos | yfinance |

A última linha diz de onde veio a cotação e o perfil. Se a Brapi não responder, a tool devolve a saída antiga, só
do yfinance, terminando com um aviso que o Consolidador lê para anotar a limitação no relatório.

In [8]:
from fluxo_financeiro.tools.financial_data import buscar_dados_acao_b3

print(buscar_dados_acao_b3.invoke("WEGE3"))

2026-11-08 13:49:01,615 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


2026-11-08 13:49:02,168 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2443 chars)


Dados para WEGE3.SA:
Empresa: WEG SA
Preço Atual: R$ 48.19 (no dia: -0.23%)
Faixa de 52 semanas: R$ 41.15 a R$ 54.53
Valor de mercado: R$ 210.0 bi
LPA: R$ 1.49
P/L:  32.3x
ROE:  29.9%
DY:   3.22%
Setor: Bens Industriais | Subsetor: Máquinas e Equipamentos
Maiores do setor na Brapi (Producer Manufacturing, por valor de mercado): WEGE3 é a 1ª de 27.
  1. WEGE3 WEG S.A.: R$ 210.0 bi
  2. FRAS3 FRASLE MOBILITY S.A.: R$ 5.9 bi
  3. SHUL4 SCHULZ S.A.: R$ 5.4 bi
  4. DXCO3 DEXCO S.A.: R$ 4.3 bi
  5. POMO4 MARCOPOLO S.A.: R$ 4.3 bi
Sumario de Negócio:
<conteudo_externo origem="brapi">
A WEG S.A. é uma companhia brasileira de equipamentos eletroeletrônicos fundada em 1961, com sede em Jaraguá do Sul (SC), e listada na B3 sob o ticker WEGE3. A empresa atua no segmento de bens de capital industriais e desenvolve, fabrica e comercializa motores elétricos, geradores, transformadores, sistemas de automação, produtos para eletrificação e tintas industriais.

A estrutura operacional da companhia é org

## O que observar

- `XXXX3` volta `False` (404): é esse caso que faz a CLI, o dashboard e o e2e recusarem o ticker sem rodar o grafo. Um `None` no lugar (sem token, Brapi fora do ar) não recusaria nada.
- A sugestão para `WEGE4` é `WEGE3`: o fracionário `WEGE3F` foi pulado. Para `XXXX3` não há sugestão, e a mensagem fica só com a primeira frase.
- O setor do perfil está em português e na classificação da B3; o da lista de pares está em inglês. São classificações diferentes, por isso o texto da tool mostra as duas.
- A conta preço / LPA com os números da Brapi deu o mesmo P/L que o yfinance informa: é a mesma definição. Nem sempre é assim: na VALE3 a conta dá perto de 123 e o yfinance diz 77, provavelmente porque cada fonte usa um lucro de 12 meses diferente. Por isso a tool só mostra o LPA quando preço / LPA fica a até 10% do P/L.
- Na lista de pares, cada empresa aparece uma vez (a Schulz como SHUL4, sem a SHUL3), e a WEGE3 é de longe a maior: o valor de mercado dela é dezenas de vezes o do segundo colocado.
- O 403 com `MODULES_NOT_AVAILABLE` é o motivo de os demonstrativos continuarem no yfinance. Rodando a mesma célula com `VALE3`, os dados viriam, porque ela é ticker de sandbox.
- Na saída da tool, o sumário chega dentro de `<conteudo_externo origem="brapi">`: continua sendo texto de terceiros, tratado como dado.
- A última linha diz "Fonte: cotação e perfil da Brapi". Se aparecer "Fonte: yfinance (a Brapi não respondeu...)", a análise seguiu com a reserva.